In [8]:
from typing import List,TypedDict
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS, Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [9]:
load_dotenv()

True

In [10]:
persist_db_name = "./chroma_vector_store"
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
def ingest_document(file_path:str, thread_id:str):
    if file_path.endswith(".pdf"):
        loader = PyPDFLoader(file_path)
    elif file_path.endswith(".txt"):
        loader = TextLoader(file_path)
    elif file_path.endswith(".csv"):
        loader = CSVLoader(file_path)
    else:
        raise ValueError("Unsupported file type")
    
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
    chunks = splitter.split_documents(docs)


    metadatas = [{"source":file_path, "thread_id": thread_id, "page": doc.metadata.get("page", 0)} for doc in chunks]

    vector_store = Chroma.from_documents(
        documents= chunks,
        embedding= embeddings,
        persist_directory= persist_db_name,
        metadatas= metadatas
    )

In [ ]:
def fetch_documents(query:str):
    try:
        vector_db = Chroma(
                embedding_function = embeddings,
                persist_directory = persist_db_name
            )
        results = vector_db.similarity_search(query=query, k=5)

        if not results:
                return "No relevant documents found in the database."

        formatted_results = "Here are the top matching documents from the database:\n\n"
        for i, doc in enumerate(results):
            # Extract metadata if available (PyPDFLoader usually adds 'source' and 'page')
            source = doc.metadata.get("source", "Unknown file")
            page = doc.metadata.get("page", "Unknown page")
                    
            formatted_results += f"--- Match {i+1} (Source: {source}, Page: {page}) ---\n"
            formatted_results += f"{doc.page_content}\n\n"
                    
        return formatted_results
    
    except Exception as e:
        return f"Error retrieving documents: {str(e)}"

In [ ]:
from langchain_community.vectorstores import FAISS
# ... after you chunk your documents ...

# Create the vector store in memory
vectorstore = FAISS.from_documents(documents=chunks, embedding=your_embedding_model)

# Save it to a folder on your computer so it doesn't disappear!
vectorstore.save_local("my_local_database")

In [ ]:
from langchain_community.vectorstores import FAISS

@tool
def query_uploaded_documents(question: str) -> str:
    """Use this tool to answer questions about the user's uploaded documents."""
    
    # 1. Load the database from your hard drive
    # Note: 'allow_dangerous_deserialization' is required in newer LangChain versions for security
    vectorstore = FAISS.load_local(
        folder_path="my_local_database", 
        embeddings=your_embedding_model, 
        allow_dangerous_deserialization=True 
    )
    
    # 2. Grab the current thread ID
    current_thread = st.session_state["current_thread"]
    
    # 3. Search using FAISS's built-in metadata filtering!
    docs = vectorstore.similarity_search(
        query=question, 
        k=4, 
        filter={"thread_id": current_thread} # FAISS handles this perfectly
    )
    
    return "\n".join([doc.page_content for doc in docs])